In [1]:
%load_ext autoreload
%autoreload 2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import pickle
import random
import time
import scimap as sm
import anndata as ad
from functools import partial
from helperFunctions import *
from smallestEnclosingCircle import make_circle
from sklearn.mixture import GaussianMixture
from ecd_helperFunctions import *

Running SCIMAP  2.1.1


### Topographical Correlation Map Feature Based Cox-PH Model

**Step 1: Data Preprocessing**
- Load data into correct format for TCM
- Divide data into ROIs and save as .hd5 files<br>

**Step 2: Generate Complete Spatial Randomness Datasets**
- Generate datasets with equal number of cells as each ROI
- Create "paired" CSR datasets for each ROI<br>

**Step 3: Calculate Topographical Correlation Maps**
- Calculate TCM for both real and CSR datasets<br>

**Step 4: Compare Positive and Negative TCM Distributions**
- For positive TCM values, use K-S test to compare real and CSR datasets
- For negative TCM values, use K-S test to compare real and CSR datasets<br>

**Step 5: Rank ROIs based on K-S Score**
- Use rank to select the top 10 ROIs with the most positive correlation and top 10 ROIs with the most negative correlation between cell markers<br>

**Step 6: Calculate the distance between the centroids of the top 10 ROIs**
- Calculate the pairwise distance between the of the top 10 postive and negative ROIs
- This will result pos_TCM_dist_12, pos_TCM_dist_23, pos_TCM_dist_34, etc. for positive TCM values and neg_TCM_dist_12, neg_TCM_dist_23, neg_TCM_dist_34, etc. for negative TCM values<br>

**Step 7: Calculate the Cox-PH model with L2 regularization**
- The model will have 29 covariates added per marker pair that is assessed in the TCM

### **STEP ONE**: Data Preprocessing

In [2]:
# Get all .csv files in one list
data_dir = '/michorlab/labsyspharm_ORION-CRC/labsyspharm_ORION-CRC/aws_data/'
data_parent_dir = os.listdir(data_dir)

file_paths = []
for i in range(len(data_parent_dir)):
    files = os.listdir(data_dir + data_parent_dir[i])
    csv_files = [file for file in files if file.endswith('.csv')]
    for file in csv_files:
        file_path = os.path.join(data_dir, data_parent_dir[i], file)
        file_paths.append(file_path)

print(file_paths)

['/michorlab/labsyspharm_ORION-CRC/labsyspharm_ORION-CRC/aws_data/CRC07/P37_S35-CRC07.csv', '/michorlab/labsyspharm_ORION-CRC/labsyspharm_ORION-CRC/aws_data/CRC35/P37_S78-CRC35.csv', '/michorlab/labsyspharm_ORION-CRC/labsyspharm_ORION-CRC/aws_data/CRC14/P37_S46-CRC14.csv', '/michorlab/labsyspharm_ORION-CRC/labsyspharm_ORION-CRC/aws_data/CRC26/P37_S63-CRC26.csv', '/michorlab/labsyspharm_ORION-CRC/labsyspharm_ORION-CRC/aws_data/CRC21/P37_S58-CRC21.csv', '/michorlab/labsyspharm_ORION-CRC/labsyspharm_ORION-CRC/aws_data/CRC19/P37_S51-CRC19.csv', '/michorlab/labsyspharm_ORION-CRC/labsyspharm_ORION-CRC/aws_data/CRC13/P37_S45-CRC13.csv', '/michorlab/labsyspharm_ORION-CRC/labsyspharm_ORION-CRC/aws_data/CRC32/P37_S75-CRC32.csv', '/michorlab/labsyspharm_ORION-CRC/labsyspharm_ORION-CRC/aws_data/CRC38/P37_S81-CRC38.csv', '/michorlab/labsyspharm_ORION-CRC/labsyspharm_ORION-CRC/aws_data/CRC33_02/P37_S76_02-CRC33_02.csv', '/michorlab/labsyspharm_ORION-CRC/labsyspharm_ORION-CRC/aws_data/CRC04/P37_S32-C

In [ ]:
# Define markers of interest (reduce the file size needed to save)
markers = ['CD45', 'CD4', 'SMA', 'PD-L1', 'Pan-CK']
grid_directory = '/michorlab/ecdyer/multiplex_spatial/crc_grid_data/cph_grids/'

##### Only run the below step if you want to generate NEW grid files for the TCM

In [ ]:
# Apply GMM to each file and save the grid
for f in file_paths:
    df = apply_gmm([f], markers)
    grid_file_name = f.split('/')[-1].split('.')[0] + '_gmm.h5'
    create_tile_rois(df, tile_size=5000, save=True, save_hdf5=grid_file_name)

### **STEP TWO/THREE**: Generate Complete Spatial Randomness Datasets and Calculate TCMs

In [ ]:
# Get all grid files
cwd = '/michorlab/ecdyer/multiplex_spatial/crc_grid_data/cph_grids/'
grid_files = [f for f in os.listdir(cwd) if f.endswith('.h5')]
